# Ñu: Lenguaje de Programación en Español



In [1]:
import os
for f in os.listdir():
    if f.startswith("ñu") or (f.endswith(".py") or f.endswith(".tokens")):
        os.remove(f)

In [2]:
!pip install -U antlr4-python3-runtime &> /dev/null

In [3]:
%%writefile ñu.g4
grammar ñu;

root: stat+ EOF;

declaracion
    : tipo ID ASIG expr
    ;

asignacion
    : ID ASIG expr
    ;

stat
    : declaracion       # ToDeclaracion
    | asignacion        # ToAsignacion
    | mostrarStat       # Print
    | ifStat            # ToIf
    | whileStat         # ToWhile
    | forStat           # ToFor
    | funcionStat       # ToFuncion
    | returnStat        # ToReturn
    | expr              # ExprStat
    ;

mostrarStat: MOSTRAR LPAREN expr RPAREN;

tipo: NUM_TIPO | TEXTO_TIPO | BOOL_TIPO | AUTO;

expr: compExpr;

compExpr: addExpr (op=(IGUAL | DIF | MENOR_IGUAL | MAYOR_IGUAL | MENOR | MAYOR) addExpr)? # Comparador
        ;

ifStat : SI expr bloque
        (ELIF expr bloque)*
        (SINO bloque)?
        # Condition
       ;

whileStat: MIENTRAS expr bloque
        # While
       ;

forInit
    : declaracion       # InitDeclaracion
    | asignacion        # InitAsignacion
    ;

forUpdate
    : asignacion        # UpdateAsignacion
    ;

forStat
    : PARA LPAREN
      forInit
      PYC
      expr
      PYC
      forUpdate
      RPAREN
      bloque
      # For
    ;

funcionStat
    : FUNCION
      ID
      LPAREN
      parametros?
      RPAREN
      bloque
      # Funcion
    ;

parametros
    : parametro (COMA parametro)*
    ;

parametro
    : tipo ID
    ;

llamadaFuncion
    : ID LPAREN argumentos? RPAREN 
    ;

argumentos
    : expr (COMA expr)*
    ;

returnStat
    : RETORNAR expr?
      # Return
    ;

bloque: LKEY stat* RKEY;

addExpr: addExpr op=(MAS | MENOS) mulExpr # AddSub
        | mulExpr                         # ToMul
        ;

mulExpr: mulExpr op=(MULT | DIV) powExpr  # MulDiv
        | powExpr              # ToPow
        ;

powExpr: <assoc=right> atom ELEVADO powExpr  # Potencia
      | atom                   # ToAtom
      ;
      

atom
    : LPAREN expr RPAREN     # Parentesis
    | NUM                    # Numero
    | STRING                 # Texto
    | BOOL                   # Booleano
    | ingresarExpr           # Input
    | llamadaFuncion         # AtomFuncion
    | ID                     # Variable
    | MENOS atom             # Negativo
    ;

ingresarExpr: INGRESAR LPAREN RPAREN;


// === Signos y palabras reservadas ===
SI: 'si';
SINO: 'sino';
ELIF: 'osino';

LKEY: '{';
RKEY: '}';

IGUAL: '==';
DIF: '!=';
MENOR_IGUAL: '<=';
MAYOR_IGUAL: '>=';
MENOR: '<';
MAYOR: '>';

MOSTRAR: 'mostrar';
INGRESAR: 'ingresar';

MIENTRAS: 'mientras';
PARA : 'para';
PYC : ';';

FUNCION: 'funcion';
COMA: ',';
RETORNAR : 'retornar';

NUM_TIPO: 'num';
TEXTO_TIPO: 'texto';
BOOL_TIPO: 'bool';
AUTO: 'auto';

BOOL: 'verdadero' | 'falso';
STRING: '"' .*? '"' | '\'' .*? '\'';
NUM : [0-9]+ ('.' [0-9]+)? ;

ID  : [a-zA-ZáéíóúÁÉÍÓÚñÑ_][a-zA-ZáéíóúÁÉÍÓÚñÑ_0-9]* ;

ASIG: '=';

LPAREN: '(';
RPAREN: ')';
MULT: '*';
DIV: '/';
MAS : '+' ;
MENOS : '-' ;
ELEVADO: '**';


WS : [ \t\r\n]+ -> skip ;

Writing ñu.g4


In [4]:
!curl -O https://www.antlr.org/download/antlr-4.13.2-complete.jar &> /dev/null

In [5]:
!java -cp .:antlr-4.13.2-complete.jar org.antlr.v4.Tool ñu.g4 -no-listener -visitor -Dlanguage=Python3

In [5]:
%%writefile EvalVisitor.py
from antlr4 import *
import operator

oper = {
    '+': operator.add,
    '-': operator.sub,
    '*': operator.mul,
    '/': operator.truediv,
    '**': operator.pow
}

rel = {
    '==': operator.eq,
    '!=': operator.ne,
    '<': operator.lt,
    '>': operator.gt,
    '<=': operator.le,
    '>=': operator.ge
}

class ReturnValue(Exception):
    def __init__(self, value):
        self.value = value

if __name__ is not None and "." in __name__:
    from .ñuParser import ñuParser
    from .ñuVisitor import ñuVisitor
else:
    from ñuParser import ñuParser
    from ñuVisitor import ñuVisitor


class EvalVisitor(ñuVisitor):
    def __init__(self):
        self.memory = {}
        self.functions = {}

    # === Helpers ===
    def inferir_tipo(self, valor):
        if isinstance(valor, bool):
            return "bool"
        elif isinstance(valor, (int, float)):
            return "num"
        elif isinstance(valor, str):
            return "texto"
        else:
            raise Exception("Tipo no soportado")

    def validar_tipo(self, tipo, valor):
        if tipo == "num":
            return isinstance(valor, (int, float))
        elif tipo == "texto":
            return isinstance(valor, str)
        elif tipo == "bool":
            return isinstance(valor, bool)
        return False

    # === Roots y bloque ===
    def visitRoot(self, ctx):
        l = list(ctx.getChildren())
        for i in range(len(l) - 1):
            result = self.visit(l[i])
        return result

    def visitBloque(self, ctx):
        l = list(ctx.getChildren())
        for child in l:
            if child.getText() not in ['{', '}']:
                result = self.visit(child)
        return result

    # === Variables ===
    def visitDeclaracion(self, ctx):
        l = list(ctx.getChildren())
        tipo = l[0].getText()
        nombre = l[1].getText()
        valor = self.visit(l[3])
        if tipo == 'auto':
            tipo = self.inferir_tipo(valor)
        if not self.validar_tipo(tipo, valor):
            raise Exception(f"Tipo incompatible para '{nombre}'")
        self.memory[nombre] = {
            'tipo': tipo,
            'valor': valor
        }
        return valor

    def visitToDeclaracion(self, ctx):
        return self.visit(ctx.declaracion())

    def visitInitDeclaracion(self, ctx):
        return self.visit(ctx.declaracion())

    def visitAsignacion(self, ctx):
        l = list(ctx.getChildren())
        nombre = l[0].getText()
        valor = self.visit(l[2])
        if nombre not in self.memory:
            raise Exception(f"Variable '{nombre}' no definida")
        tipo = self.memory[nombre]['tipo']
        if not self.validar_tipo(tipo, valor):
            raise Exception(f"Tipo incompatible para '{nombre}'")
        self.memory[nombre]['valor'] = valor
        return valor

    def visitToAsignacion(self, ctx):
        return self.visit(ctx.asignacion())

    def visitInitAsignacion(self, ctx):
        return self.visit(ctx.asignacion())

    def visitUpdateAsignacion(self, ctx):
        return self.visit(ctx.asignacion())

    def visitVariable(self, ctx):
        nombre = ctx.getText()
        if nombre in self.memory:
            return self.memory[nombre]['valor']
        raise Exception(f"Variable '{nombre}' no definida")

    # === Valores ===
    def visitNumero(self, ctx):
        texto = ctx.getText()
        return float(texto) if '.' in texto else int(texto)

    def visitTexto(self, ctx):
        texto = ctx.getText()
        return texto[1:-1]

    def visitBooleano(self, ctx):
        return True if ctx.getText() == 'verdadero' else False

    # === Expresiones matematicas y comparaciones ===
    def visitAddSub(self, ctx):
        l = list(ctx.getChildren())
        left = self.visit(l[0])
        right = self.visit(l[2])
        op = l[1].getText()

        if op == '+': # -> para concatenar strings
            if isinstance(left, str) or isinstance(right, str):
                return str(left) + str(right)

        return oper[op](left, right)

    def visitMulDiv(self, ctx):
        l = list(ctx.getChildren())
        return oper[l[1].getText()](
            self.visit(l[0]),
            self.visit(l[2])
        )

    def visitPotencia(self, ctx):
        l = list(ctx.getChildren())
        return oper[l[1].getText()](
            self.visit(l[0]),
            self.visit(l[2])
        )

    def visitComparador(self, ctx):
        l = list(ctx.getChildren())
        if len(l) == 1:
            return self.visit(l[0])
        return int(
            rel[l[1].getText()](
                self.visit(l[0]),
                self.visit(l[2])
            )
        )

    def visitParentesis(self, ctx):
        return self.visit(ctx.expr())

    def visitNegativo(self, ctx):
        return -self.visit(ctx.atom())

    # === Condicionales y bucles ===
    def visitCondition(self, ctx):
        l = list(ctx.getChildren())
        if self.visit(l[1]) == 1:
            return self.visit(l[2])
        index = 3
        while index < len(l):
            texto = l[index].getText()
            if texto == 'osino':
                condicion = l[index + 1]
                bloque = l[index + 2]
                if self.visit(condicion) == 1:
                    return self.visit(bloque)
                index += 3
            elif texto == 'sino':
                return self.visit(l[index + 1])
            else:
                index += 1

    def visitWhile(self, ctx):
        l = list(ctx.getChildren())
        condicion = l[1]
        bloque = l[2]
        while self.visit(condicion):
            self.visit(bloque)

    def visitFor(self, ctx):
        self.visit(ctx.forInit())
        while self.visit(ctx.expr()):
            self.visit(ctx.bloque())
            self.visit(ctx.forUpdate())

    # === Input / Output ===
    def visitPrint(self, ctx):
        print(self.visit(ctx.mostrarStat().expr()))

    def visitIngresarExpr(self, ctx):
        texto = input()
        try:
            return float(texto) if '.' in texto else int(texto)
        except:
            return texto
    
    # === Funciones ===
    def visitToFuncion(self, ctx):
        return self.visit(ctx.funcionStat())
        
    def visitFuncion(self, ctx):
        nombre = ctx.ID().getText()
        parametros = []

        if ctx.parametros():
            for p in ctx.parametros().parametro():
                tipo = p.tipo().getText()
                nombre_param = p.ID().getText()

                parametros.append(
                    (tipo, nombre_param)
                )

        self.functions[nombre] = {
            "params": parametros,
            "block": ctx.bloque()
        }

    def visitLlamadaFuncion(self, ctx):
        nombre = ctx.ID().getText()
        if nombre not in self.functions:
            raise Exception(f"Función {nombre} no definida")

        funcion = self.functions[nombre]

        argumentos = []
        if ctx.argumentos():
            for expr in ctx.argumentos().expr():
                argumentos.append(self.visit(expr))

        if len(argumentos) != len(funcion["params"]):
            raise Exception("Cantidad incorrecta de argumentos")

        old_memory = self.memory.copy()

        for (tipo,nombre),valor in zip(funcion["params"], argumentos):
            self.memory[nombre] = {
                "tipo": tipo,
                "valor": valor
            }

        try: 
            self.visit(funcion["block"])
            retorno = None
        except ReturnValue as rv:
            retorno = rv.value

        self.memory = old_memory
        return retorno

    def visitReturn(self, ctx):
        if ctx.expr():
            valor = self.visit(ctx.expr())
        else: 
            valor = None
            
        raise ReturnValue(valor)

Writing EvalVisitor.py


# Explicacion Visitor

## VisitCondition

Explicacion del visitcondition

```
l = [
    token_SI,          # Índice 0: "si"
    ctx_expr_x_5,      # Índice 1: expresión (x > 5)
    ctx_bloque_1,      # Índice 2: bloque { mostrar("Mayor"); }
    token_ELIF_1,      # Índice 3: "osino"
    ctx_expr_x_5_2,    # Índice 4: expresión (x == 5)
    ctx_bloque_2,      # Índice 5: bloque { mostrar("Igual"); }
    token_SINO,        # Índice 6: "sino"
    ctx_bloque_3       # Índice 7: bloque { mostrar("Menor"); }
]
```

## VisitFor
```
0  para
1  (
2  num
3  i
4  =
5  0
6  ;
7  i < 5
8  ;
9  i
10 =
11 i + 1
12 )
13 bloque
```

# Ejemplos de Codigo

In [25]:
%%writefile ejemplito1.ñu
num x = 2
auto y = x + 1
mostrar(y)

Overwriting ejemplito1.ñu


In [16]:
%%writefile ejemplito2.ñu
num edad = ingresar()

si edad > 18 {
  mostrar("Mayor de edad")
}
osino edad == 18 {
  mostrar('Es 18 justo')
}
sino {
  mostrar("Es menor de edad")
}

Writing ejemplito2.ñu


In [17]:
%%writefile ejemplito3.ñu
num x = 0

mientras x < 5 {
    mostrar(x)
    x = x + 1
}

Writing ejemplito3.ñu


In [18]:
%%writefile ejemplito4.ñu
para (num i = 5; i > 0; i = i - 1) {
    mostrar(i)
}

Writing ejemplito4.ñu


In [19]:
%%writefile ejemplito5.ñu
funcion saludar(texto nombre){
    mostrar("Hola, " + nombre)
}

saludar("José")

Writing ejemplito5.ñu


In [20]:
%%writefile ejemplito6.ñu
funcion suma(num a, num b){
    retornar a + b
}

mostrar(suma(5, 5))

Writing ejemplito6.ñu


In [21]:
%%writefile ejemplito7.ñu
funcion suma(num a, num b){
    retornar a + b
}

num x = suma(3, 4)
mostrar(x)

Writing ejemplito7.ñu


In [6]:
%%writefile ejemplito8.ñu
funcion duplicar_hasta(num num_base, num limite){
    si (num_base > limite){
        mostrar("El numero base no debe ser mayor al numero limite")
        retornar 0
    }

    num cont = 0
    mientras (num_base < limite) {
        num_base = num_base * 2
        mostrar(num_base)
        cont = cont + 1
    }

    mostrar("Numero total de duplicaciones: " + cont)
    retornar num_base
}

num numero_limite = 8
auto duplicaciones = duplicar_hasta(2, numero_limite)
mostrar("Resultado final: " + duplicaciones + ". Que es igual al numero limite " + numero_limite)

Writing ejemplito8.ñu


In [7]:
from antlr4 import *
from ñuLexer import ñuLexer
from ñuParser import ñuParser
from EvalVisitor import EvalVisitor

input_stream = FileStream("ejemplito8.ñu", encoding="utf-8")
lexer = ñuLexer(input_stream)
token_stream = CommonTokenStream(lexer)
parser = ñuParser(token_stream)
tree = parser.root()
# print(tree.toStringTree(recog=parser))

visitor = EvalVisitor()
visitor.visitRoot(tree)

4
8
Numero total de duplicaciones: 2
Resultado final: 8. Que es igual al numero limite 8


# LLVM Visitor
Visitor con LLVM usando `llvmlite`. 

Es diferente al Visitor anterior ya que este utiliza modulos propios de LLVM y además genera un código de representación intermedia (IR) que luego correremos por separado con `lli`

In [8]:
!pip show antlr4-python3-runtime

Name: antlr4-python3-runtime
Version: 4.13.2
Summary: ANTLR 4.13.2 runtime for Python 3
Home-page: http://www.antlr.org
Author: Terence Parr, Sam Harwell
Author-email: Eric Vergnaud <eric.vergnaud@wanadoo.fr>
License: BSD
Location: /usr/local/lib/python3.12/dist-packages
Requires: 
Required-by: omegaconf


In [6]:
!pip install antlr4-python3-runtime llvmlite

!wget https://www.antlr.org/download/antlr-4.13.2-complete.jar

--2026-07-04 14:56:13--  https://www.antlr.org/download/antlr-4.13.2-complete.jar
Resolving www.antlr.org (www.antlr.org)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to www.antlr.org (www.antlr.org)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2140045 (2.0M) [application/java-archive]
Saving to: ‘antlr-4.13.2-complete.jar.2’

antlr-4.13.2-comple 100%[===================>]   2.04M  --.-KB/s    in 0.06s   

2026-07-04 14:56:13 (34.1 MB/s) - ‘antlr-4.13.2-complete.jar.2’ saved [2140045/2140045]



In [7]:
%%writefile LLVMVisitor.py
from llvmlite import ir
from ñuParser import ñuParser
from ñuVisitor import ñuVisitor

class LLVMVisitor(ñuVisitor):
    def __init__(self):
        self.module = ir.Module(name="Ñu")
        self.builder = None
        self.function = None
        self.variables = {}

        self.printf = None
        self.declare_printf()

    # Para funcion mostrar()
    def declare_printf(self):
        voidptr_ty = ir.IntType(8).as_pointer()

        printf_ty = ir.FunctionType(
            ir.IntType(32),
            [voidptr_ty],
            var_arg=True
        )

        self.printf = ir.Function(
            self.module,
            printf_ty,
            name="printf"
        )
    
    # Genera la funcion main de IR:
    # define i32 @main(){}
    def visitRoot(self, ctx):
        main_type = ir.FunctionType(
            ir.IntType(32),
            []
        )

        self.function = ir.Function(
            self.module,
            main_type,
            "main"
        )

        block = self.function.append_basic_block("entry")
        self.builder = ir.IRBuilder(block)

        for stmt in ctx.stat():
            self.visit(stmt)

        self.builder.ret(ir.Constant(ir.IntType(32), 0))
        return self.module
    
    def visitDeclaracion(self, ctx):
        nombre = ctx.ID().getText()
        valor = self.visit(ctx.expr())

        ptr = self.builder.alloca(
            valor.type,
            name=nombre
        )

        self.builder.store(valor, ptr)
        self.variables[nombre] = ptr

        return valor

    def visitAsignacion(self, ctx):
        nombre = ctx.ID().getText()
        valor = self.visit(ctx.expr())
        ptr = self.variables[nombre]

        self.builder.store(valor, ptr)
        return valor
    
    def visitVariable(self, ctx):
        nombre = ctx.getText()
        ptr = self.variables[nombre]

        return self.builder.load(ptr)
    
    def visitNumero(self, ctx):
        texto = ctx.getText()

        if "." in texto:

            return ir.Constant(
                ir.DoubleType(),
                float(texto)
            )

        return ir.Constant(
            ir.IntType(32),
            int(texto)
        )

    def visitNegativo(self, ctx):
        valor = self.visit(ctx.atom())
        cero = ir.Constant(valor.type,0)

        return self.builder.sub(cero, valor)

    def visitParentesis(self, ctx):
        return self.visit(ctx.expr())
    
    # === Operaciones matematicas y comparaciones ===
    def visitAddSub(self, ctx):
        left = self.visit(ctx.addExpr())
        right = self.visit(ctx.mulExpr())

        op_type = ctx.op.type

        if op_type == ñuParser.MAS:
            return self.builder.add(left, right, name="tmp_add")
        elif op_type == ñuParser.MENOS:
            return self.builder.sub(left, right, name="tmp_sub")
        
    def visitMulDiv(self, ctx):
        left = self.visit(ctx.mulExpr())
        right = self.visit(ctx.powExpr())

        op_type = ctx.op.type

        if op_type == ñuParser.MULT:
            return self.builder.mul(left, right, name="tmp_mul")
        elif op_type == ñuParser.DIV:
            return self.builder.sdiv(left, right, name="tmp_div")

    def visitComparador(self, ctx):
        if ctx.op is None:
            return self.visit(ctx.addExpr(0))

        left = self.visit(ctx.addExpr(0))
        right = self.visit(ctx.addExpr(1))

        op = ctx.op.text
        return self.builder.icmp_signed(op, left, right, "cmp")


Writing LLVMVisitor.py


In [8]:
%%writefile ejemplitollvm01.ñu
num x = 10
num y = 20

num z = x + y

Overwriting ejemplitollvm01.ñu


In [10]:
from antlr4 import *
from ñuLexer import ñuLexer
from ñuParser import ñuParser
from LLVMVisitor import LLVMVisitor

nombre_archivo = "ejemplitollvm01.ñu"
input_stream = FileStream(nombre_archivo, encoding="utf-8")

lexer = ñuLexer(input_stream)
token_stream = CommonTokenStream(lexer)
parser = ñuParser(token_stream)
tree = parser.root()
visitor = LLVMVisitor()
module = visitor.visit(tree)

archivo_ll = nombre_archivo.replace(".ñu", ".ll")
with open(archivo_ll, "w") as f:
    f.write(str(module))

print("Archivo " + archivo_ll + " generado.")

Archivo ejemplitollvm01.ll generado.
